In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold,cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.decomposition import PCA

In [2]:
df=pd.read_csv('dataset/gurgaon_properties_post_feature_selection_v2.csv')

In [3]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 66,3.98,3,4,3+,Moderately Old,2200.0,1,0,1,Medium,Low Floor
1,flat,sector 62,2.15,2,2,3,Under Construction,1676.0,0,0,0,Medium,High Floor
2,flat,sector 65,2.44,3,3,2,Under Construction,1654.0,0,0,0,Low,Mid Floor
3,flat,sector 50,3.10,3,4,3+,Moderately Old,2450.0,1,0,1,High,Mid Floor
4,flat,sector 33,1.55,3,4,1,Relatively New,1421.0,1,0,0,Low,Low Floor


In [4]:
df['furnishing_type'].value_counts()

furnishing_type
0    2334
1    1033
2     187
Name: count, dtype: int64

In [5]:
df['furnishing_type'] = df['furnishing_type'].replace({0.0:'unfurnished',1.0:'semifurnished',2.0:'furnished'})

In [6]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 66,3.98,3,4,3+,Moderately Old,2200.0,1,0,semifurnished,Medium,Low Floor
1,flat,sector 62,2.15,2,2,3,Under Construction,1676.0,0,0,unfurnished,Medium,High Floor
2,flat,sector 65,2.44,3,3,2,Under Construction,1654.0,0,0,unfurnished,Low,Mid Floor
3,flat,sector 50,3.10,3,4,3+,Moderately Old,2450.0,1,0,semifurnished,High,Mid Floor
4,flat,sector 33,1.55,3,4,1,Relatively New,1421.0,1,0,unfurnished,Low,Low Floor


In [7]:
X=df.drop(columns=['price'])
y=df['price']

In [8]:
y_transformed=np.log1p(y)

Ordinal Encoding

In [9]:
columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_encode)
    ], 
    remainder='passthrough'
)

In [11]:
pipeline=Pipeline([
    ('preprocessor',preprocessor),
    ('regressor',LinearRegression())
])

In [12]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [13]:
scores.mean(),scores.std()

(np.float64(0.7382658970453898), np.float64(0.02879332836656098))

In [14]:
X_train,X_test,y_train,y_test=train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [15]:
pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](12,)","['property_type','sector','bedRoom',...,'furnishing_type', 'luxury_category','floor_category']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,12
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed thro

In [16]:
y_pred=pipeline.predict(X_test)

In [17]:
y_pred=np.expm1(y_pred)

In [18]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.8618458626001145

In [19]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [22]:
pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   - -------------------------------------- 2.1/48.9 MB 10.3 MB/s eta 0:00:05
   ---- ----------------------------------- 5.2/48.9 MB 12.3 MB/s eta 0:00:04
   -------- ------------------------------- 10.5/48.9 MB 16.4 MB/s eta 0:00:03
   ------------- -------------------------- 16.0/48.9 MB 18.9 MB/s eta 0:00:02
   ----------------- ---------------------- 22.0/48.9 MB 20.6 MB/s eta 0:00:02
   ----------------------- ---------------- 28.6/48.9 MB 22.5 MB/s eta 0:00:01
   ---------------------------- ----------- 35.4/48.9 MB 23.9 MB/s eta 0:00:01
   ---------------------------------- ----- 41.9/48.9 MB 24.7 MB/s eta 0:00:01
   ---------------------------------------  48.0/48.9 MB 25.3 MB/s eta 0:00:01
   ---------------------------------------  48.8/48.9 MB 25.5 MB/s eta 0:00:01
   ---------------------------------------- 48.9/48.9 MB 22.8 MB/s  0:00

In [23]:
# Linear Models
from sklearn.linear_model import Lasso, Ridge

# Decision Tree Model
from sklearn.tree import DecisionTreeRegressor

# Tree Ensembles
from sklearn.ensemble import (
    AdaBoostRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    RandomForestRegressor,
)

# Neural Network
from sklearn.neural_network import MLPRegressor

# Gradient Boosted Trees (requires: pip install xgboost)
from xgboost import XGBRegressor

In [24]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [25]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [26]:
model_output

[['linear_reg', np.float64(0.7382658970453898), 0.8618458626001145],
 ['svr', np.float64(0.7601472604359036), 0.8485913820325941],
 ['ridge', np.float64(0.7382685560745593), 0.8619525981269053],
 ['LASSO', np.float64(0.05741865475937528), 1.6041933531895347],
 ['decision tree', np.float64(0.788133410990672), 0.7456702417617205],
 ['random forest', np.float64(0.8826079287381594), 0.5224879241363346],
 ['extra trees', np.float64(0.8662816519889514), 0.5473663737840682],
 ['gradient boosting', np.float64(0.8756921784041196), 0.5738151448242211],
 ['adaboost', np.float64(0.7648111846933456), 0.8118339256326423],
 ['mlp', np.float64(0.8057394065924643), 0.7328010173671837],
 ['xgboost', np.float64(0.8914003896374286), 0.5477733926481335]]

In [27]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [28]:
model_df.sort_values(['mae'])

,name,r2,mae
5,random forest,0.882608,0.522488
6,extra trees,0.866282,0.547366
10,xgboost,0.891400,0.547773
7,gradient boosting,0.875692,0.573815
9,mlp,0.805739,0.732801
4,decision tree,0.788133,0.745670
8,adaboost,0.764811,0.811834
1,svr,0.760147,0.848591
0,linear_reg,0.738266,0.861846
2,ridge,0.738269,0.861953


OneHotEncoding

In [36]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), columns_to_encode),
        ('cat1', OneHotEncoder(drop='first', handle_unknown='infrequent_if_exist'), ['sector', 'agePossession', 'furnishing_type'])
    ], 
    remainder='passthrough'
)

In [37]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [38]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

c:\Users\HEMANT\OneDrive\Desktop\Capstone_Project\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
c:\Users\HEMANT\OneDrive\Desktop\Capstone_Project\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)


In [39]:
scores.mean(),scores.std()

(np.float64(0.8564353475705921), np.float64(0.0208418563857779))

In [40]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [41]:
pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](12,)","['property_type','sector','bedRoom',...,'furnishing_type', 'luxury_category','floor_category']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,12
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed

In [42]:
y_pred = pipeline.predict(X_test)

In [43]:
y_pred=np.expm1(y_pred)

In [44]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.6246467214315766

In [45]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [46]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [47]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

c:\Users\HEMANT\OneDrive\Desktop\Capstone_Project\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
c:\Users\HEMANT\OneDrive\Desktop\Capstone_Project\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
c:\Users\HEMANT\OneDrive\Desktop\Capstone_Project\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as the infrequent category.
  warnings.warn(msg, UserWarning)
c:\Users\HEMANT\OneDrive\Desktop\Capstone_Project\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown

In [48]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [49]:
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.892534,0.452845
5,random forest,0.890850,0.479556
10,xgboost,0.899179,0.529507
7,gradient boosting,0.878346,0.553891
4,decision tree,0.812010,0.570154
9,mlp,0.875195,0.598337
0,linear_reg,0.856435,0.624647
2,ridge,0.856754,0.625583
1,svr,0.764854,0.841522
8,adaboost,0.761232,0.850688


OneHotEncoding With PCA

In [50]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room'],
        ),
        (
            'cat',
            OrdinalEncoder(
                handle_unknown='use_encoded_value', unknown_value=-1
            ),
            columns_to_encode,
        ),
        (
            'cat1',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False),
            ['sector', 'agePossession'],
        ),
    ],
    remainder='drop',
    sparse_threshold=0,  # Ensures the output matrix passed to PCA is 100% dense
)

In [51]:
pipeline = Pipeline(
    [
        ('preprocessor', preprocessor),
        ('scaler_all', StandardScaler()),  
        ('pca', PCA(n_components=0.95)),
        ('regressor', LinearRegression()),
    ]
)

In [52]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [53]:
scores.mean(),scores.std()

(np.float64(0.8077138448864829), np.float64(0.026679203693741157))

In [54]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=0.95)),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [55]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [56]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [57]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [58]:
model_df.sort_values(['mae'])

,name,r2,mae
5,random forest,0.751219,0.749423
6,extra trees,0.727142,0.783608
4,decision tree,0.679621,0.857716
10,xgboost,0.614119,0.997843
7,gradient boosting,0.611981,1.059225
8,adaboost,0.305468,1.416515
1,svr,0.219952,1.432151
9,mlp,0.217935,1.503316
3,LASSO,0.057627,1.604105
2,ridge,0.060514,1.610058


Target Encoder

In [59]:
!pip install category_encoders

In [60]:
import category_encoders as ce

columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']
ordinal_cols = [
    'property_type',
    'balcony',
    'furnishing_type',
    'luxury_category',
    'floor_category',
]
num_cols = ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        (
            'cat_ord',
            OrdinalEncoder(
                handle_unknown='use_encoded_value', unknown_value=-1
            ),
            ordinal_cols,
        ),
        (
            'cat_ohe',
            OneHotEncoder(
                drop='first',
                handle_unknown='infrequent_if_exist',
                sparse_output=False,
            ),
            ['agePossession'],
        ),
        ('target_enc', ce.TargetEncoder(), ['sector']),
    ],
    remainder='drop',
)

In [61]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [62]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [63]:
scores.mean(),scores.std()

(np.float64(0.8298502781464936), np.float64(0.018679727615119976))

In [64]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [65]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [66]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [67]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [68]:
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.891273,0.493479
5,random forest,0.893028,0.496303
10,xgboost,0.898034,0.530093
7,gradient boosting,0.884220,0.559577
1,svr,0.861389,0.594759
9,mlp,0.851230,0.617746
4,decision tree,0.816619,0.681445
0,linear_reg,0.829850,0.689426
2,ridge,0.829868,0.689875
8,adaboost,0.816674,0.752522
